# 12 · Multi-Agent Patterns

*Four ways to arrange several agents, and how to tell which one you have.*

"Multi-agent system" describes a category, not a design. Once you have more
than one agent, the question is how control moves between them — and there are
a handful of standard answers, each with a different failure mode.

This notebook works through four:

| Pattern | Who decides what happens next |
|---|---|
| **Supervisor** | one agent, every time |
| **Plan-and-Execute** | a plan made up front, then followed |
| **Reflexion** | a critic, after the work is done |
| **Swarm / Handoff** | whichever agent currently holds control |

We identify which of these the Trip Concierge actually uses — by reading the
code rather than the README — and run the ones that are implemented.

> **Prerequisite.** `docker compose up -d`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json, pathlib, httpx

AGENTS = {
    "flight-agent": 8010, "hotel-agent": 8011, "itinerary-agent": 8012,
    "critic-agent": 8015, "todo-agent": 8016, "research-agent": 8018,
}
PLANNER = "http://localhost:8001"

up = sum(1 for p in AGENTS.values()
         if httpx.get(f"http://localhost:{p}/health", timeout=3).status_code == 200)
print(f"{up}/{len(AGENTS)} sub-agents reachable")
print("planner:", httpx.get(f"{PLANNER}/health", timeout=5).json()["tools"], "tools")

## Pattern 1 — Supervisor

One agent owns the decisions. Workers do as they are told and report back; they
never choose what happens next.

The test for this pattern is simple: **can a worker start work on its own?**
If every path runs through one agent, it is a supervisor.

In [ ]:
# Who in this codebase is able to delegate to another agent?
import subprocess
hits = [h for h in subprocess.run(
    ["grep", "-rl", "--include=*.py", "delegate_to_a2a_agent", "services/"],
    capture_output=True, text=True).stdout.strip().splitlines()
    if "__pycache__" not in h]

print("files that can call another agent:")
for h in hits:
    print("   ", h)

print("\nservices/ directory:")
for d in sorted(p.name for p in pathlib.Path("services").iterdir() if p.is_dir()):
    print("   ", d)

Only the planner. Six worker agents exist, and not one of them holds an A2A
client or a registry of peers.

Look at what a worker actually gets bound to its model:

In [ ]:
src = pathlib.Path("services/flight_agent/main.py").read_text()
for line in src.splitlines():
    s = line.strip()
    if "MultiServerMCPClient" in s or "get_tools()" in s or "bind_tools" in s:
        print("   ", s)

`flight-agent` binds only what `mcp-airline` offers. It has no
`delegate_to_hotel_agent`, because it has no way to reach another agent at all.

So the shape is strictly one-directional:

```
                    planner  (decides everything)
                   /   |   |   |   |   \
             flight hotel itin critic todo research
```

**This is a Supervisor**, and the demonstration is structural — the workers
could not hand off even if their prompts told them to.

## Pattern 2 — Plan-and-Execute (what this is *not*)

The README calls the planner a *"Plan-and-Execute supervisor"*. That phrase is
worth testing, because the two patterns behave very differently when something
goes wrong.

**Plan-and-Execute** means: produce a complete plan first, then carry out its
steps. There is a planning phase, and the plan is an object you can inspect.

Look at the graph's nodes.

In [ ]:
src = pathlib.Path("services/planner/graph.py").read_text()
for line in src.splitlines():
    s = line.strip()
    if s.startswith("_graph.add_node") or s.startswith("_graph.add_edge") \
       or s.startswith("_graph.add_conditional_edges"):
        print("   ", s)

In [ ]:
# Is there any node that produces a plan?
print("nodes containing 'plan':",
      [l.strip() for l in src.splitlines() if "add_node" in l and "plan" in l.lower()] or "none")

# Where does the ordering actually live?
prompt = pathlib.Path("services/planner/prompt.py").read_text()
steps = [l.strip() for l in prompt.splitlines()
         if l.strip()[:2].rstrip(".").isdigit() and "delegate_to" in l]
print(f"\nordered steps written into the system prompt: {len(steps)}")
for s in steps[:4]:
    print("   ", s[:88])

There is no planning node. The graph is `model → tools → model`, looping — the
same two-node cycle from notebook 07. The planner decides **one step at a time**,
choosing its next tool from whatever the last result was.

The ordering that looks like a plan is **prose in the system prompt**. It is
guidance the model usually follows, not a structure the graph enforces.

> **Why the distinction matters.** In real Plan-and-Execute, a failed step is
> caught against the plan — you know step 4 of 7 failed and what was supposed to
> follow. Here, a failure just becomes another observation the model reacts to.
> That is more adaptive and less predictable, and it means you cannot show a
> user "3 of 7 steps complete" without inventing it.

So the honest label is **Supervisor with a ReAct loop**, not Plan-and-Execute.

## Pattern 3 — Reflexion (critic-driven revision)

A second agent reviews the first agent's output and sends back a critique. The
work is then revised. This is the one pattern here that is genuinely implemented
as its own agent.

In [ ]:
card = httpx.get("http://localhost:8015/.well-known/agent-card.json", timeout=10).json()
print("name       :", card["name"])
for s in card["skills"]:
    print("skill      :", s["id"])
    print("            ", s["description"][:150])

Let us actually use it. We send the critic a deliberately flawed itinerary and
read its verdict.

In [ ]:
A2A_HEADERS = {"Content-Type": "application/json", "A2A-Version": "1.0"}

def a2a(base: str, brief: str) -> dict:
    envelope = {
        "jsonrpc": "2.0", "id": "nb12", "method": "SendMessage",
        "params": {"message": {"messageId": "m-nb12", "role": "ROLE_USER",
                               "parts": [{"text": brief}]}},
    }
    r = httpx.post(base, headers=A2A_HEADERS, json=envelope,
                   timeout=httpx.Timeout(connect=5, read=180, write=30, pool=10))
    r.raise_for_status()
    return r.json()["result"]["task"]

BAD_PLAN = (
    "Review this Tokyo itinerary for 2 adults, 15-19 Oct 2026. "
    "Day 1: land at 07:55, then Tsukiji market, teamLab, Shibuya, Senso-ji, "
    "a day trip to Hakone, and a 20:00 dinner in Ginza. "
    "Day 2: free. Day 3: free. Day 4: fly home at 09:00, "
    "with a 08:30 breakfast booking in Shinjuku."
)

task = a2a("http://localhost:8015/", BAD_PLAN)
critique = "".join(p.get("text", "") for p in task["status"]["message"]["parts"])
print(critique[:1100])

That is Reflexion working: a separate model, with its own prompt, looking for
problems the original author had no incentive to find.

The pattern's cost is visible too — that was a whole extra model call, and in the
full flow the itinerary agent then has to revise and be re-reviewed. Reflexion
buys quality with latency.

In `prompt.py` the planner is told to call the critic *after* the itinerary
agent, which is why review happens at all. Nothing in the graph enforces it.

## Pattern 4 — Swarm / Handoff (not implemented here)

In a swarm, agents hand control **directly to each other**. The flight agent
finishes and hands to the hotel agent; no supervisor is involved.

The README lists this pattern. The code does not contain it.

In [ ]:
import subprocess
out = subprocess.run(["grep", "-rniE", "handoff|hand-off|swarm",
                      "--include=*.py", "--include=*.md", "--include=*.yml",
                      "services/", "shared/", "README.md", "docker-compose.yml"],
                     capture_output=True, text=True).stdout
hits = [l for l in out.splitlines() if "__pycache__" not in l]
print(f"{len(hits)} match(es) across services/, shared/ and the compose file:\n")
for h in hits:
    print("   ", h[:110])

One match, and it is the README claiming the pattern.

### What it would take, and what it would cost

To make this a swarm, every worker would need an A2A client and a registry of
its peers — roughly what `dynamic_a2a_tools.py` does for the planner, repeated
six times.

The harder problem is not the plumbing. It is that **this system has rules that
only hold because one agent owns the sequence**:

- the ordering in `prompt.py` — search before hold, check budget before commit;
- the HITL gate on `capture_payment` (notebook 11).

That gate lives in the planner's graph. In a swarm there is no single graph the
payment must pass through, so an agent could reach payment by a route that never
crosses the interrupt. You would have to re-implement the gate in every agent
that might touch money, and trust all of them.

> **A supervisor is a chokepoint, and chokepoints are how you enforce things.**
> That is the real trade-off: swarms are more flexible and much harder to
> constrain.

## Choosing between them

| | Supervisor | Plan-and-Execute | Reflexion | Swarm |
|---|---|---|---|---|
| Control | one agent | the plan | critic loop | whoever holds it |
| Adapts mid-run | yes | poorly | yes | yes |
| Progress is inspectable | no | **yes** | no | no |
| Easy to gate/audit | **yes** | yes | yes | **no** |
| Cost | 1 call/step | cheap after planning | **2× or more** | varies |
| Bottleneck | the supervisor | rigid plan | latency | coordination |

They compose. This system is a **supervisor** with **reflexion** layered in for
itinerary review — which is a common and sensible combination.

## Recap

1. **Supervisor** is what the Trip Concierge is. Proven structurally: only the
   planner can delegate, and workers hold no peer tools.
2. **Plan-and-Execute** it is not, despite the README. There is no planning node
   — the ordering is prose in the system prompt, and the graph is a ReAct loop.
   The practical consequence is that you have no inspectable plan to report
   progress against.
3. **Reflexion** is genuinely implemented as `critic-agent`, and costs an extra
   model call per review.
4. **Swarm/Handoff** appears once in the repository, in the README's claim.
   Implementing it would mean giving up the chokepoint that makes the payment
   gate enforceable.
5. **Patterns compose**, and the choice is mostly about where you need control:
   a supervisor gives you somewhere to put the rules.

---

### Exercises

1. Add a second reviewer with a different lens — cost rather than feasibility —
   and send it the same flawed itinerary. Where do the two critiques disagree?
2. Give `flight-agent` a `delegate_to_hotel_agent` tool. What has to change, and
   what happens to the payment gate?
3. Make the plan explicit: add a node that writes an ordered step list into state
   before any delegation. What can the UI show now that it could not before?
4. The critic is called because the prompt says so. Remove that instruction and
   run an itinerary request. Does review still happen?